In [0]:
import pandas as pd

files = [
{"file": "map_cities"},
{"file":"map_cancellation_reasons"},
{"file":"map_payment_methods"},
{"file":"map_ride_statuses"},
{"file":"map_vehicle_makes"},
{"file":"map_vehicle_types"}
]

for file in files:

    url = f"https://dluberprojectdev.blob.core.windows.net/raw/ingestion/{file['file']}.json?<your-token>"

    df = pd.read_json(url)
    df_spark = spark.createDataFrame(df)
    
    # Writing Data To the Bronze Layer
    df_spark.write.format("delta")\
            .mode("overwrite")\
            .option("overwriteSchema", "true")\
            .saveAsTable(f"uber.bronze.{file['file']}")

In [0]:
url = "https://dluberprojectdev.blob.core.windows.net/raw/ingestion/bulk_rides.json?<your-token>"

df = pd.read_json(url)
df_spark = spark.createDataFrame(df)
if not spark.catalog.tableExists("uber.bronze.bulk_rides"):
    df_spark.write.format("delta")\
            .mode("overwrite")\
            .saveAsTable(f"uber.bronze.bulk_rides")
    print("This will not run more than 1 time")

In [0]:
%sql
SELECT * FROM uber.bronze.map_cities

city_id,city,state,region,updated_at
1,New York,NY,Northeast,2026-02-28T16:06:33.718Z
2,Los Angeles,CA,West,2026-02-28T16:06:33.718Z
3,Chicago,IL,Midwest,2026-02-28T16:06:33.718Z
4,Houston,TX,South,2026-02-28T16:06:33.718Z
5,Phoenix,AZ,Southwest,2026-02-28T16:06:33.718Z
6,Philadelphia,PA,Northeast,2026-02-28T16:06:33.718Z
7,San Antonio,TX,South,2026-02-28T16:06:33.718Z
8,San Diego,CA,West,2026-02-28T16:06:33.718Z
9,Dallas,TX,South,2026-02-28T16:06:33.718Z
10,San Jose,CA,West,2026-02-28T16:06:33.718Z


In [0]:

%sql 
SELECT * FROM uber.bronze.rides_raw


key,value,topic,partition,offset,timestamp,timestampType,rides
null,eyJyaWRlX2lkIjogImU3ZDIxYmMxLTk3OTEtNDBlYS05NjU0LTA4MGJiZWQzZmJlNSIsICJjb25maXJtYXRpb25fbnVtYmVyIjogInFwOS04MDYyLWZ6NzQiLCAicGFzc2VuZ2VyX2lkIjogImM1ZGMxMGU5LWEwODktNDI5ZS1iMzBiLWEzODJmZmRhODVkZSIsICJkcml2ZXJfaWQiOiAiODg5YmFmMGUtNmMxOC00MDBjLThhMTYtMTg0MjA2YmViNjZiIiwgInZlaGljbGVfaWQiOiAiNGI3NjIwMGMtYjA0NC00OTE0LWFmYmQtZDE0NjkxNTU4Y2IwIiwgInBpY2t1cF9sb2NhdGlvbl9pZCI6ICJkOWUwNDk5NS00ZGY3LTQyYTUtOTNhNC1mZmEyYWVlNzE3NzgiLCAiZHJvcG9mZl9sb2NhdGlvbl9pZCI6ICJjOTRkNTVmZC0zNjgxLTRhZTQtYjMwMC05ZTMzNzNhY2E5N2IiLCAidmVoaWNsZV90eXBlX2lkIjogMywgInZlaGljbGVfbWFrZV9pZCI6IDEsICJwYXltZW50X21ldGhvZF9pZCI6IDEsICJyaWRlX3N0YXR1c19pZCI6IDEsICJwaWNrdXBfY2l0eV9pZCI6IDMsICJkcm9wb2ZmX2NpdHlfaWQiOiA3LCAiY2FuY2VsbGF0aW9uX3JlYXNvbl9pZCI6IDQsICJwYXNzZW5nZXJfbmFtZSI6ICJEYXJyZWxsIERhbHRvbiIsICJwYXNzZW5nZXJfZW1haWwiOiAic3VzYW5nb256YWxlc0BleGFtcGxlLmNvbSIsICJwYXNzZW5nZXJfcGhvbmUiOiAiMzgzLjg5Mi41NzY4eDQ4MzUiLCAiZHJpdmVyX25hbWUiOiAiSm9zaHVhIENvb3BlciIsICJkcml2ZXJfcmF0aW5nIjogNC43OCwgImRyaXZlcl9waG9uZSI6ICIrMS0yODUtMjIxLTg5NjJ4NTg2NyIsICJkcml2ZXJfbGljZW5zZSI6ICJqSy14Q2MtMDM1MTQ4MiIsICJ2ZWhpY2xlX21vZGVsIjogIlBvc3NpYmxlIiwgInZlaGljbGVfY29sb3IiOiAiR3JheSIsICJsaWNlbnNlX3BsYXRlIjogInJMUy0yNTU4IiwgInBpY2t1cF9hZGRyZXNzIjogIjQxODAgUmFjaGVsIFN0YXRpb24gQXB0LiA3ODQsIEVhc3QgRGFuaWVsbGVib3JvdWdoLCBBWiAxODYwNyIsICJwaWNrdXBfbGF0aXR1ZGUiOiAtNzQuNjIzNDE5LCAicGlja3VwX2xvbmdpdHVkZSI6IDExOC4wMDY1NzUsICJkcm9wb2ZmX2FkZHJlc3MiOiAiUFNDIDg3MjIsIEJveCA4Nzk0LCBBUE8gQUUgNDgwNzUiLCAiZHJvcG9mZl9sYXRpdHVkZSI6IC04Mi41OTI0NDgsICJkcm9wb2ZmX2xvbmdpdHVkZSI6IC05OS4wNDAzODcsICJkaXN0YW5jZV9taWxlcyI6IDMzLjM2LCAiZHVyYXRpb25fbWludXRlcyI6IDUsICJib29raW5nX3RpbWVzdGFtcCI6ICIyMDI2LTAyLTIxVDE5OjM1OjU4LjE4NjEzMCIsICJwaWNrdXBfdGltZXN0YW1wIjogIjIwMjYtMDItMjFUMTk6NDA6NTguMTg2MTMwIiwgImRyb3BvZmZfdGltZXN0YW1wIjogIjIwMjYtMDItMjFUMTk6NDU6NTguMTg2MTMwIiwgImJhc2VfZmFyZSI6IDIuNSwgImRpc3RhbmNlX2ZhcmUiOiA1OC4zOCwgInRpbWVfZmFyZSI6IDEuNzUsICJzdXJnZV9tdWx0aXBsaWVyIjogMS4yNCwgInN1YnRvdGFsIjogNzcuNjYsICJ0aXBfYW1vdW50IjogNS4wMiwgInRvdGFsX2ZhcmUiOiA4Mi42OCwgInJhdGluZyI6IDR9,ubertopic,0,0,2026-02-27T16:40:58.234Z,0,"{""ride_id"": ""e7d21bc1-9791-40ea-9654-080bbed3fbe5"", ""confirmation_number"": ""qp9-8062-fz74"", ""passenger_id"": ""c5dc10e9-a089-429e-b30b-a382ffda85de"", ""driver_id"": ""889baf0e-6c18-400c-8a16-184206beb66b"", ""vehicle_id"": ""4b76200c-b044-4914-afbd-d14691558cb0"", ""pickup_location_id"": ""d9e04995-4df7-42a5-93a4-ffa2aee71778"", ""dropoff_location_id"": ""c94d55fd-3681-4ae4-b300-9e3373aca97b"", ""vehicle_type_id"": 3, ""vehicle_make_id"": 1, ""payment_method_id"": 1, ""ride_status_id"": 1, ""pickup_city_id"": 3, ""dropoff_city_id"": 7, ""cancellation_reason_id"": 4, ""passenger_name"": ""Darrell Dalton"", ""passenger_email"": ""susangonzales@example.com"", ""passenger_phone"": ""383.892.5768x4835"", ""driver_name"": ""Joshua Cooper"", ""driver_rating"": 4.78, ""driver_phone"": ""+1-285-221-8962x5867"", ""driver_license"": ""jK-xCc-0351482"", ""vehicle_model"": ""Possible"", ""vehicle_color"": ""Gray"", ""license_plate"": ""rLS-2558"", ""pickup_address"": ""4180 Rachel Station Apt. 784, East Danielleborough, AZ 18607"", ""pickup_latitude"": -74.623419, ""pickup_longitude"": 118.006575, ""dropoff_address"": ""PSC 8722, Box 8794, APO AE 48075"", ""dropoff_latitude"": -82.592448, ""dropoff_longitude"": -99.040387, ""distance_miles"": 33.36, ""duration_minutes"": 5, ""booking_timestamp"": ""2026-02-21T19:35:58.186130"", ""pickup_timestamp"": ""2026-02-21T19:40:58.186130"", ""dropoff_timestamp"": ""2026-02-21T19:45:58.186130"", ""base_fare"": 2.5, ""distance_fare"": 58.38, ""time_fare"": 1.75, ""surge_multiplier"": 1.24, ""subtotal"": 77.66, ""tip_amount"": 5.02, ""total_fare"": 82.68, ""rating"": 4}"
null,eyJyaWRlX2lkIjogIjc4OTk4NGFiLTQxZTgtNDQzNi1hZjY5LTU1YTdjZDhiNzRmMiIsICJjb25maXJtYXRpb25fbnVtYmVyIjogIkZpMS0xNzUwLVN5MDkiLCAicGFzc2VuZ2VyX2lkIjogImQwOTM3MWJmLThiMDUtNDdlMy05OGMwLTE2NDgyYmRjY2I4NCIsICJkcml2ZXJfaWQiOiAiMTUwZWFlYjQtMmM2MC00Nzg4LWI1NTk